## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# Initialize the Groq model
model = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0,
)
agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

**Capital of the Moon: Selene‑Prime**

---

### Overview
Selene‑Prime is the bustling, domed metropolis that serves as the political, cultural, and economic heart of humanity’s lunar federation. Perched in the Mare Imbrium basin, the city is a marvel of engineering and a symbol of humanity’s resolve to thrive beyond Earth.

---

## 1. Location & Layout

| Feature | Details |
|---------|---------|
| **Site** | A 150‑km‑wide crater floor in Mare Imbrium, chosen for its relatively flat basaltic plains and proximity to the historic Luna 2 landing site. |
| **Terrains** | Three concentric zones: the **Inner Dome** (government & ceremonial), the **Mid‑Ring** (residential, commercial, research), and the **Outer Belt** (industrial, logistics, launch pads). |
| **Elevation** | The city sits 12 m above the regolith, supported by a network of **regolith‑reinforced polymer arches** and **maglev levitation pads** that counteract lunar dust settlement. |
| **Transportation** | A **maglev loop** circ

## Image input

In [6]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [7]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x0000026232FC8F40>, 'last_modified': datetime.datetime(2026, 10, 23, 10, 25, 51, 329000, tzinfo=datetime.timezone.utc)},)


In [8]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

BadRequestError: Error code: 400 - {'error': {'message': 'messages[1].content must be a string', 'type': 'invalid_request_error', 'param': 'messages[1].content'}}

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)